## Purpose

Aligns with Epic 1 validation user stories: compare **synthetic** `manifest.csv` to **real** statistics from `analyze_real_voids.py` using Kolmogorov–Smirnov tests on void counts and area fractions.

The same logic lives in `scripts/validation/compare_distributions.py`.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

from udm_epic1.dataset.pipeline import DatasetPipeline, PipelineConfig

In [ ]:
# Small synthetic run → manifest.csv; compare first column stats to a toy "real" sample
with tempfile.TemporaryDirectory() as tmp:
    cfg = PipelineConfig.from_yaml("../configs/default.yaml")
    cfg.total_images = 40
    cfg.num_workers = 2
    cfg.output_dir = tmp
    pipeline = DatasetPipeline(cfg)
    manifest_path = pipeline.run()

    df = pd.read_csv(manifest_path)
    s_n = df["n_voids"].to_numpy(dtype=float)
    s_a = df["total_void_area_fraction"].to_numpy(dtype=float)

    # Toy "real" arrays (same length for a meaningful KS demo — use real JSON in production)
    r_n = s_n + np.random.randn(len(s_n)) * 0.5
    r_a = s_a + np.random.randn(len(s_a)) * 0.01

    ks_n, p_n = stats.ks_2samp(s_n, r_n)
    ks_a, p_a = stats.ks_2samp(s_a, r_a)
    print(f"n_voids: KS={ks_n:.4f}, p={p_n:.4g}")
    print(f"area fraction: KS={ks_a:.4f}, p={p_a:.4g}")

## CLI (production)

```bash
python scripts/validation/compare_distributions.py \
  --manifest data/synthetic/manifest.csv \
  --real-stats results/void_statistics.json \
  --output results/distribution_report.md
```

Also: `udm-generate validate` and `udm-generate stats` check filesystem layout and manifest summaries.